# DelayShield 02 - Model development

Feature engineering, the chronological split, the six trained configurations,
threshold tuning, the cost model and SHAP. The same code paths the application
uses at runtime.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('../src'))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import train as T
from feature_engineering import FEATURE_COLUMNS, NUMERIC_FEATURES, CATEGORICAL_FEATURES, TARGET
pd.set_option('display.width', 160)

## 1. Time-aware features

`_lagged_outcome_stats` computes each group's history using only orders whose
`order_delivered_customer_date` precedes the current `order_purchase_timestamp`
- outcomes that were genuinely known at confirmation time.

In [ ]:
data = pd.read_csv('../data/processed/model_dataset.csv', parse_dates=['order_purchase_timestamp'])
print(data.shape)
print(len(NUMERIC_FEATURES), 'numeric +', len(CATEGORICAL_FEATURES), 'categorical features')
data[['seller_hist_orders','seller_hist_delay_rate','seller_orders_last_30_days',
      'lead_time_slack_days','distance_per_promised_day','delivery_delay']].describe().round(3)

### Sanity check: the lagged seller delay rate separates outcomes

In [ ]:
data.groupby('delivery_delay')[['seller_hist_delay_rate','lead_time_slack_days',
                                'distance_km','estimated_lead_time_days']].mean().round(3)

## 2. Chronological split

In [ ]:
train, val, test = T.chronological_split(data)
for name, part in [('train', train), ('val', val), ('test', test)]:
    print(f"{name:<6}{len(part):>7,}  {part['order_purchase_timestamp'].min():%Y-%m-%d} -> "
          f"{part['order_purchase_timestamp'].max():%Y-%m-%d}  late={part[TARGET].mean():.3%}")

## 3. Model comparison

Three families (logistic regression, random forest, XGBoost) x two imbalance
treatments (balanced class weights vs threshold tuning only). SMOTE is avoided
on purpose: it interpolates between orders from different time periods.

In [ ]:
comparison = pd.read_csv('../reports/model_comparison.csv')
comparison[['model','split','roc_auc','pr_auc','precision','recall','f1','brier']].round(4)

In [ ]:
meta = json.load(open('../models/model_metadata.json'))
print('selected:', meta['selected_model'], '|', meta['selection_criterion'])
print('F1 threshold:', meta['threshold_f1'], '| cost-optimal threshold:', meta['threshold_cost_optimal'])
print('risk bands:', meta['risk_thresholds'])
meta['test_metrics_at_f1_threshold']

### Retrain the selected configuration inline (optional, ~20 s)

In [ ]:
X_tr, y_tr = train[FEATURE_COLUMNS], train[TARGET]
X_va, y_va = val[FEATURE_COLUMNS], val[TARGET]
X_te, y_te = test[FEATURE_COLUMNS], test[TARGET]
model = T.build_models(pos_weight=1.0, weighted=False)['random_forest']
model.fit(X_tr, y_tr)
p_va = model.predict_proba(X_va)[:,1]; p_te = model.predict_proba(X_te)[:,1]
thr = T.tune_threshold(y_va, p_va, 'f1')
print('tuned threshold:', thr)
T.evaluate(y_te, p_te, thr)

## 4. Calibration and the probability range

In [ ]:
from sklearn.calibration import calibration_curve
pt, pp = calibration_curve(y_te, p_te, n_bins=10, strategy='quantile')
plt.figure(figsize=(5,5))
plt.plot(pp, pt, 'o-', label='model'); plt.plot([0,pp.max()],[0,pp.max()],'--',color='grey',label='ideal')
plt.xlabel('mean predicted probability'); plt.ylabel('observed delay rate')
plt.legend(); plt.title('Calibration - test window'); plt.show()
print('probability range on test:', round(p_te.min(),3), '->', round(p_te.max(),3))

## 5. Business cost model and threshold choice

All costs are ASSUMPTIONS - Olist publishes none. The comparison and its
sensitivity carry the meaning, not the absolute number.

In [ ]:
import simulation as S
sweep = S.threshold_sweep(y_te, p_te, grid=np.arange(0.06, 0.28, 0.02))
display(sweep.round(3))
print('break-even precision:', round(S.break_even_precision(), 4))
sweep.plot(x='threshold', y='saving_vs_reactive', marker='o', figsize=(9,4),
           title='Saving vs reactive policy by threshold'); plt.grid(alpha=.3); plt.show()

## 6. SHAP explanations

In [ ]:
import explain as E
imp = E.global_importance(300).head(15)
imp.set_index('feature')['mean_abs_shap'].sort_values().plot(
    kind='barh', figsize=(8,6), color='#2563eb', title='Global feature importance (mean |SHAP|)')
plt.show()
imp

In [ ]:
import predict as P
snap = P.history_snapshot()
row = P.build_feature_row(seller_id=snap['seller_ids'][5], product_category='bed_bath_table',
                          total_order_value=350.0, total_freight_value=45.0, item_count=2,
                          customer_state='AM', seller_state='SP',
                          order_date=pd.Timestamp('2018-08-15'), estimated_lead_time_days=20)
prob = float(P.predict_proba(row)[0])
print('delay probability:', round(prob, 4), '->', P.risk_level(prob))
E.top_factors(row, 5)['all'].head(10)

## 7. Conclusion

The selected configuration ranks delay risk usefully (test ROC-AUC ~0.73,
PR-AUC ~2.2x the base rate) but individual alerts are low precision, so
DelayShield is positioned as a triage and prioritisation system: predict,
explain, prioritise suppliers, recommend an action, and simulate the policy
before committing to it.